In [9]:
import cv2
import os

# -------------------------------------------------
# Settings
# -------------------------------------------------

GESTURES = ["up", "down", "left", "right"]

SAVE_DIR = "dataset"

ROI_X1, ROI_Y1 = 200, 100
ROI_X2, ROI_Y2 = 500, 400
IMG_SIZE = 128

# -------------------------------------------------
# Init folders
# -------------------------------------------------

for g in GESTURES:
    os.makedirs(os.path.join(SAVE_DIR, g), exist_ok=True)

# -------------------------------------------------
# State
# -------------------------------------------------

current_gesture = GESTURES[0]
counts = {g: len(os.listdir(os.path.join(SAVE_DIR, g))) for g in GESTURES}

cap = cv2.VideoCapture(0)

print("Keys:")
print("1..4 : switch gesture")
print("s    : save sample")
print("ESC  : exit")

# -------------------------------------------------
# Main loop
# -------------------------------------------------
 
while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # draw ROI box
    cv2.rectangle(
        frame,
        (ROI_X1, ROI_Y1),
        (ROI_X2, ROI_Y2),
        (0, 255, 0),
        2
    )

    # UI text
    cv2.putText(
        frame,
        f"Gesture: {current_gesture}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Saved: {counts[current_gesture]}",
        (10, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    cv2.putText(
        frame,
        "1-4 switch | s save | ESC exit",
        (10, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    cv2.imshow("CNN data collection", frame)

    key = cv2.waitKey(1) & 0xFF

    # -------------------------------------------------
    # Switch gesture with keys 1..4
    # -------------------------------------------------

    if key >= ord('1') and key <= ord(str(len(GESTURES))):
        idx = key - ord('1')
        current_gesture = GESTURES[idx]
        print("Switched to:", current_gesture)

    # -------------------------------------------------
    # Save sample
    # -------------------------------------------------

    if key == ord('s'):

        roi = frame[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]
        roi = cv2.resize(roi, (IMG_SIZE, IMG_SIZE))

        fname = os.path.join(
            SAVE_DIR,
            current_gesture,
            f"{counts[current_gesture]}.jpg"
        )

        cv2.imwrite(fname, roi)

        counts[current_gesture] += 1

        print("Saved:", fname)

    # -------------------------------------------------
    # Exit
    # -------------------------------------------------

    if key == 27:
        break


cap.release()
cv2.destroyAllWindows()


Keys:
1..4 : switch gesture
s    : save sample
ESC  : exit


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# -----------------------------
# Config
# -----------------------------
DATASET_DIR = "dataset"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 20

# -----------------------------
# Load dataset
# -----------------------------
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("cnn_label_classes_lrud_v2.npy", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)

# -----------------------------
# Data augmentation
# -----------------------------
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
])

# -----------------------------
# Model
# -----------------------------
inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

# -----------------------------
# Compile
# -----------------------------
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# -----------------------------
# Train
# -----------------------------
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# -----------------------------
# Save
# -----------------------------
model.save("cnn_gesture_lrud_v2.keras")


Found 291 files belonging to 7 classes.
Using 233 files for training.
Found 291 files belonging to 7 classes.
Using 58 files for validation.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,584,775 (17.49 MB)

 Trainable params: 4,584,775 (17.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 224ms/step - accuracy: 0.3777 - loss: 1.6362 - val_accuracy: 0.3276 - val_loss: 1.5644
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 174ms/step - accuracy: 0.4034 - loss: 1.3694 - val_accuracy: 0.3276 - val_loss: 1.3470
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step - accuracy: 0.3906 - loss: 1.3826 - val_accuracy: 0.3276 - val_loss: 1.2961
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step - accuracy: 0.4120 - loss: 1.3098 - val_accuracy: 0.3966 - val_loss: 1.2934
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step - accuracy: 0.4936 - loss: 1.2282 - val_accuracy: 0.3276 - val_loss: 1.2320
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step - accuracy: 0.4850 - loss: 1.2305 - val_accuracy: 0.5172 - val_loss: 1.1566
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 171ms/step - accuracy: 0.5708 - loss: 1.0972 - val_accuracy: 0.5690 - val_loss: 1.0459
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step - accuracy: 0.5536 - loss: 1.0759 - val_accuracy: 0.5517 - val_loss:

In [ ]:
import cv2
import numpy as np
from tensorflow import keras

MODEL_PATH   = "cnn_gesture_lrud_v2.keras"
CLASSES_PATH = "cnn_label_classes_lrud_v2.npy"

IMG_SIZE = (128, 128)

ROI_X1, ROI_Y1 = 200, 100
ROI_X2, ROI_Y2 = 500, 400

# -------------------------
# Load model and labels
# -------------------------
model = keras.models.load_model(MODEL_PATH)
classes = np.load(CLASSES_PATH, allow_pickle=True)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

print("Press ESC to exit")

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # draw ROI
    cv2.rectangle(frame, (ROI_X1, ROI_Y1), (ROI_X2, ROI_Y2), (0,255,0), 2)

    roi = frame[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]

    if roi.size != 0:

        img = cv2.resize(roi, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype("float32") 

        x = np.expand_dims(img, axis=0)

        probs = model.predict(x, verbose=0)[0]

        idx  = int(np.argmax(probs))
        pred = str(classes[idx])
        conf = float(probs[idx])

        cv2.putText(frame, f"Pred: {pred}", (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

        cv2.putText(frame, f"Conf: {conf:.2f}", (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)

    cv2.putText(frame, "Show hand inside box | ESC to exit",
                (10, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    cv2.imshow("CNN Gesture Test", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


SyntaxError: invalid syntax (473216908.py, line 69)

In [2]:
import tensorflow as tf

print("TF:", tf.__version__)
print("Keras path:", tf.keras.__file__)


TF: 2.20.0


ImportError: Keras cannot be imported. Check that it is installed.